In [1]:
import pickle
import numpy as np
import pandas as pd

In [2]:
class LightFM4Rec:
    def __init__(
        self,
        model,
        user_mapping,
        item_mapping,
        inv_user_mapping,
        inv_item_mapping,
        user_col,
        item_col,
        rating_col,
    ):
        self.model = model
        self.user_mapping = user_mapping
        self.item_mapping = item_mapping
        self.inv_user_mapping = inv_user_mapping
        self.inv_item_mapping = inv_item_mapping
        self.user_col = user_col
        self.item_col = item_col
        self.rating_col = rating_col

    def fit(
        self,
        rating_matrix,
        train_interactions,
        user_features=None,
        item_features=None,
        epochs=16,
    ):
        self.user_features = user_features
        self.item_features = item_features
        self.train_interactions = train_interactions
        self.model.fit(
            rating_matrix,
            user_features=self.user_features,
            item_features=self.item_features,
            epochs=epochs,
        )

    def _get_lfm_pred(self, user_id):
        pred = self.model.predict(
            user_ids=user_id,
            item_ids=self.item_ids,
            user_features=self.user_features,
            item_features=self.item_features,
        )
        return pred

    def predict(self, test, interaction_matrix, user_col, filter_seen=True, k=10):
        user_ids = test[self.user_col].map(self.user_mapping).unique()
        self.item_ids = list(self.item_mapping.values())

        pred = pd.DataFrame(user_ids, columns=[user_col])
        scores = np.vstack(pred[user_col].apply(self._get_lfm_pred).values)

        if filter_seen:
            scores += np.nan_to_num(interaction_matrix.todense()[user_ids] * -np.inf)

        ind_part = np.argpartition(scores, -k)[:, -k:].copy()
        scores_not_sorted = np.take_along_axis(scores, ind_part, axis=1)
        ind_sorted = np.argsort(scores_not_sorted, axis=1)
        scores_sorted = np.sort(scores_not_sorted, axis=1)
        indices = np.take_along_axis(ind_part, ind_sorted, axis=1)

        preds = pd.DataFrame(
            {
                self.user_col: user_ids,
                self.item_col: np.flip(indices, axis=1).tolist(),
                self.rating_col: np.flip(scores_sorted, axis=1).tolist(),
            }
        ).explode([self.item_col, self.rating_col])
        preds[self.user_col] = preds[self.user_col].map(self.inv_user_mapping)
        preds[self.item_col] = preds[self.item_col].map(self.inv_item_mapping)

        return preds

In [3]:
with open("lightfm_sber_model.pkl", "rb") as f:
    model = pickle.load(f)

/Users/antonshishkov/Projects/diploma/.venv/lib/python3.14/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [5]:
model.predict(
    test=pd.DataFrame({"user_name": ["financial_economist_and_analyst"]}),
    interaction_matrix=model.train_interactions,
    user_col="user_name",
    filter_seen=False,
    k=10,
)

,user_name,item_title,rating
0,financial_economist_and_analyst,Мониторинг макроэкономической и финансовой сит...,1.345695
0,financial_economist_and_analyst,Общественное восприятие экономических показате...,0.939642
0,financial_economist_and_analyst,Динамика валютного курса рубля: факторы и возд...,0.749359
0,financial_economist_and_analyst,Факторы и источники инфляции в России,0.700904
0,financial_economist_and_analyst,Анализ больших данных для поддержки принятия р...,0.48729
0,financial_economist_and_analyst,Теория и практика валютной политики в России,0.455178
0,financial_economist_and_analyst,Каналы денежной трансмиссии и их особенности в...,0.454207
0,financial_economist_and_analyst,Финансовые инновации в монетарной политике России,0.440754
0,financial_economist_and_analyst,Экономические циклы в российской экономике: пр...,0.439816
0,financial_economist_and_analyst,"Антиинфляционная политика: теория, инструменты...",0.418933
